# **Experiment Notebook**



In [ ]:
# Do not modify this code
!pip install -q utstd

from utstd.ipyrenders import *

In [ ]:
# Do not modify this code
import warnings
warnings.simplefilter(action='ignore')

In [ ]:
import pandas as pd

pd.__version__

---
## Student Information

In [ ]:
student_name = "Nandeeswari Kumaravelu"
student_id = "25980436"

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBRegressor, XGBClassifier

from nba_draft_features.aggregation import aggregate_daily
from nba_draft_features.targets import add_cci, add_whc, shift_target

---
## A. Project Description

In [ ]:
business_use_case_description = """
Open-Meteo is expanding its weather platform with AI-driven predictive
services. This project delivers a Climate Comfort Index (CCI) forecasting
service for Sydney: given weather observations up to a given date, the model
predicts how comfortable outdoor conditions will be 3 days ahead, on a 0-100
scale, deployed as a REST API for downstream applications.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_use_case_description', value=business_use_case_description)

In [ ]:
business_objectives = """
Accurate CCI forecasts help event organisers, tourism operators, and outdoor
recreation businesses plan activities around comfortable conditions, reducing
last-minute cancellations and improving customer experience. Incorrect
predictions mainly cause planning inconvenience rather than safety risk, so
the cost of error here is primarily reputational and operational.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_objectives', value=business_objectives)

In [ ]:
stakeholders_expectations_explanations = """
Primary users are event planners, outdoor recreation and tourism businesses,
and residents accessing the public API (GET /predict/index/comfort_climate).
Predictions inform day-to-day activity planning on a 3-day horizon. Anyone
scheduling outdoor activities in Sydney is a direct beneficiary; incorrect
predictions mostly affect convenience and trust rather than safety.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='stakeholders_expectations_explanations', value=stakeholders_expectations_explanations)

---
## C. Data Understanding

### C.1   Load Raw Data

In [ ]:
daily = pd.read_csv('../../data/processed/sydney_daily_targets.csv', parse_dates=['date'])
print(daily.shape)
display(daily.head())
daily.info()

### C.2 Define Target variable

In [ ]:
# The Climate Comfort Index (CCI) is defined by the AT2 spec as a weighted
# sum of five clipped component scores derived from daily-aggregated weather
# variables. See nba_draft_features.targets.add_cci for the implementation.
print('CCI = 100 * (0.35*TempScore + 0.20*HumidityScore + 0.15*WindScore + 0.15*CloudScore + 0.15*RainScore)')

In [ ]:
target_definition_explanations = """
CCI is a regression target combining five component scores (temperature,
humidity, wind, cloud cover, precipitation), each clipped to [0, 1] and
weighted per the AT2 specification, then scaled to 0-100. Weights reflect
relative importance to outdoor comfort: temperature dominates (0.35),
followed by humidity (0.20), with wind/cloud/rain each at 0.15. The model
predicts CCI 3 days ahead, so each row's label is the CCI value shifted
3 days into the future relative to that row's date.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_definition_explanations', value=target_definition_explanations)

### C.3 Create Target variable

In [ ]:
target_name = 'cci_target_3d'

daily = add_cci(daily)
daily[target_name] = shift_target(daily, 'cci', horizon_days=3, date_col='date')
daily[['date', 'cci', target_name]].head(10)

### C.4 Explore Target variable

In [ ]:
daily['cci'].hist(bins=30)
plt.title('CCI Distribution'); plt.xlabel('CCI (0-100)'); plt.show()
daily['cci'].describe()

In [ ]:
daily.set_index('date')['cci'].plot(figsize=(14,4), title='CCI over time')
plt.ylabel('CCI'); plt.show()

In [ ]:
target_distribution_explanations = """
CCI ranges from roughly 29 to 99 across the dataset, with a mean around 71
and a left-skewed distribution -- consistent with Sydney's generally mild
climate, where most days fall in the "Comfortable" to "Excellent" bands. The
lower tail reflects occasional extreme-heat, high-humidity, or high-rain days
that sharply reduce comfort. No missing values after aggregation; the final
rows have a missing target (no future value exists yet) and are dropped
before modelling.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_distribution_explanations', value=target_distribution_explanations)

### C.5 Explore Feature of Interest `temperature_2m`

In [ ]:
daily['temperature_2m'].plot(figsize=(14,4), title='Daily mean temperature'); plt.show()
print('Correlation with CCI:', daily['temperature_2m'].corr(daily['cci']))

In [ ]:
daily['temperature_2m'].hist(bins=30); plt.title('Temperature distribution'); plt.show()

In [ ]:
feature_1_insights = """
temperature_2m shows the strongest individual correlation with CCI, matching
its 0.35 weighting in the formula. Sydney's daily mean temperature varies
seasonally and is the single largest driver of day-to-day comfort variation.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### C.6 Explore Feature of Interest `relative_humidity_2m`

In [ ]:
daily['relative_humidity_2m'].plot(figsize=(14,4), title='Daily mean humidity'); plt.show()
print('Correlation with CCI:', daily['relative_humidity_2m'].corr(daily['cci']))

In [ ]:
feature_2_insights = """
Humidity shows a moderate, non-linear relationship with CCI -- the formula
penalises deviation from 50% in either direction, so very dry and very humid
days both reduce comfort. Sydney's humidity is fairly stable year-round
compared to temperature, so it contributes less day-to-day variance overall.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### C.7 Explore Feature of Interest `precipitation`

In [ ]:
daily['precipitation'].plot(figsize=(14,4), title='Daily precipitation'); plt.show()
print('Correlation with CCI:', daily['precipitation'].corr(daily['cci']))

In [ ]:
daily['precipitation'].hist(bins=30); plt.title('Precipitation distribution (right-skewed)'); plt.show()

In [ ]:
feature_n_insights = """
precipitation is heavily right-skewed -- most days have zero or near-zero
rainfall, punctuated by occasional large spikes, matching Sydney's pattern of
long dry spells interrupted by intense rain events. Its correlation with CCI
is weaker linearly than temperature's, but the formula's clipped RainScore
means heavy-rain days still sharply reduce comfort regardless of linear
correlation strength.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

### C.n Explore Feature of Interest `\<put feature name here\>`

> You can add more cells related to other feeatures in this section

---
## D. Feature Selection



### D.1 Approach "Correlation analysis"

In [ ]:
corr_cols = ['temperature_2m','relative_humidity_2m','wind_speed_10m','cloud_cover','precipitation','cci']
corr_matrix = daily[corr_cols].corr()
corr_matrix['cci'].sort_values(ascending=False)

In [ ]:
plt.imshow(corr_matrix, cmap='coolwarm')
plt.xticks(range(len(corr_cols)), corr_cols, rotation=90)
plt.yticks(range(len(corr_cols)), corr_cols)
plt.colorbar(); plt.title('Correlation matrix'); plt.show()

In [ ]:
feature_selection_1_insights = """
Correlation analysis confirms temperature_2m and cloud_cover are the
strongest linear drivers of CCI, consistent with their formula weights.
Wind and precipitation show weaker linear correlation individually but
contribute via the formula's clipped, non-linear component scores, so linear
correlation alone understates their real importance.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### D.2 Approach "Baseline feature importance"

In [ ]:
from xgboost import XGBRegressor as _XGBR
_baseline = _XGBR(n_estimators=100, max_depth=4, random_state=42)
_X = daily[corr_cols[:-1]].fillna(daily[corr_cols[:-1]].mean())
_y = daily['cci']
_baseline.fit(_X, _y)
pd.Series(_baseline.feature_importances_, index=corr_cols[:-1]).sort_values(ascending=False)

In [ ]:
feature_selection_2_insights = """
A quick baseline XGBoost regressor's feature importances broadly agree with
the correlation analysis, confirming temperature and cloud cover as the
dominant predictors, with humidity, wind, and precipitation contributing
secondary but non-trivial importance -- supporting keeping all five formula
inputs as base features rather than dropping any.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### D.n Approach "\<describe_approach_here\>"

> You can add more cells related to other approaches in this section

## D.z Final Selection of Features

In [ ]:
features_list = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'cloud_cover', 'precipitation']
features_list

In [ ]:
feature_selection_explanations = """
All five raw weather variables used directly in the CCI formula are retained
as base features, since the target is a deterministic function of exactly
these inputs (plus a 3-day time shift). Both correlation and baseline
importance analysis support keeping all five; engineered lag/rolling/
calendar features (Section F) are added on top to help the model learn
short-term persistence and seasonality that the raw formula does not
directly capture.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## E. Data Preparation

### E.1 Data Transformation "Missing value check"

In [ ]:
print(daily.isna().sum())

In [ ]:
data_cleaning_1_explanations = """
Aggregating from hourly to daily (mean/sum/max per variable, as specified in
the AT2 brief) already removes most within-day noise and missingness. The only
remaining NaNs are in the target column for the final few rows, where no future
value exists yet for the shift horizon -- these are structural, not data-quality
issues, and are handled by dropping (Section E.3), not imputing.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### E.2 Data Transformation "Date continuity check"

In [ ]:
full_range = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')
missing_dates = full_range.difference(daily['date'])
print('Missing calendar days:', len(missing_dates))

In [ ]:
data_cleaning_2_explanations = """
Since shift_target() assumes no gaps in the daily calendar (a gap would shift
the horizon across missing days incorrectly), this check confirms whether the
Open-Meteo pull produced a complete daily series. Any missing dates would need
to be reindexed and interpolated or flagged before the horizon-shift step is
trusted.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### E.3 Data Transformation "Drop rows with missing target"

In [ ]:
daily_clean = daily.dropna(subset=[target_name]).reset_index(drop=True)
print('Rows before:', len(daily), '| Rows after dropping missing target:', len(daily_clean))

In [ ]:
data_cleaning_3_explanations = """
Rows with a missing target (the last few days of the dataset, where no future
value exists yet at the required horizon) are dropped entirely rather than
imputed, since the target is a real future observation and cannot be estimated
without leaking information or fabricating labels.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### E.n Fixing "\<describe_issue_here\>"

> You can add more cells related to other issues in this section

---
## F. Feature Engineering

### F.1 New Feature "temp_lag1 (yesterday's temperature)"

In [ ]:
daily_clean['temp_lag1'] = daily_clean['temperature_2m'].shift(1)

In [ ]:
feature_engineering_1_explanations = """
temp_lag1 captures short-term weather persistence -- comfort conditions
rarely change drastically from one day to the next, so yesterday's
temperature is informative for predicting the near-future comfort trend.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### F.2 New Feature "temp_roll7 (7-day rolling mean temperature)"

In [ ]:
daily_clean['temp_roll7'] = daily_clean['temperature_2m'].rolling(7).mean()

In [ ]:
feature_engineering_2_explanations = """
temp_roll7 captures the current seasonal baseline, helping the model
distinguish an unusually hot day in winter from an ordinary one in summer --
information the raw daily temperature alone does not convey.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### F.3 New Feature "cci_lag1 and calendar features"

In [ ]:
daily_clean['cci_lag1'] = daily_clean['cci'].shift(1)
daily_clean['month'] = daily_clean['date'].dt.month
daily_clean['day_of_year'] = daily_clean['date'].dt.dayofyear
daily_clean = daily_clean.dropna().reset_index(drop=True)
daily_clean.tail()

In [ ]:
feature_engineering_n_explanations = """
cci_lag1 (yesterday's realised CCI) is a strong persistence signal in its
own right. month and day_of_year explicitly encode seasonality, since
Sydney's comfort patterns follow a strong annual cycle that raw weather
variables alone do not fully capture for a model with no other time context.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

### F.n Fixing "\<describe_issue_here\>"

> You can add more cells related to new features in this section

---
## G. Data Preparation for Modeling

### G.1 Split Datasets

In [ ]:
# Chronological split (critical for time series -- avoids leaking future
# information into training via lag/rolling features)
n = len(daily_clean)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = daily_clean.iloc[:train_end]
val_df = daily_clean.iloc[train_end:val_end]
test_df = daily_clean.iloc[val_end:]
print('Train:', len(train_df), '| Val:', len(val_df), '| Test:', len(test_df))

In [ ]:
data_splitting_explanations = """
A chronological (time-ordered) 70/15/15 split is used instead of a random
split, since this is time series data with lag/rolling features -- a random
split would leak information from "future" rows into training via those lag
features, producing unrealistically optimistic validation/test performance.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### G.2 Data Transformation "Feature/target matrices"

In [ ]:
engineered_cols = ['temp_lag1', 'temp_roll7', 'cci_lag1', 'month', 'day_of_year']
feature_cols = features_list + engineered_cols

X_train, y_train = train_df[feature_cols], train_df[target_name]
X_val, y_val = val_df[feature_cols], val_df[target_name]
X_test, y_test = test_df[feature_cols], test_df[target_name]

In [ ]:
data_transformation_1_explanations = """
Feature and target matrices are built from the chronologically-split train/val/
test dataframes, combining the five spec-mandated raw variables with the
engineered lag/rolling/calendar features from Section F.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### G.3 Data Transformation "Scaling check"

In [ ]:
# XGBoost is tree-based and invariant to monotonic feature scaling, so no
# explicit min-max or standard scaling is required here.
print('Feature ranges (train):')
X_train.describe().loc[['min','max']]

In [ ]:
data_transformation_2_explanations = """
No feature scaling (min-max or standardisation) is applied, since the chosen
algorithm (XGBoost) is tree-based and splits on raw feature values -- scaling
has no effect on a tree-based model's learned splits or performance, unlike
distance-based or gradient-descent-based algorithms where it would matter.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### G.4 Data Transformation "Final NaN check"

In [ ]:
print('NaNs in X_train:', X_train.isna().sum().sum())
print('NaNs in X_val:', X_val.isna().sum().sum())
print('NaNs in X_test:', X_test.isna().sum().sum())

In [ ]:
data_transformation_3_explanations = """
A final NaN check confirms all lag/rolling engineered features have valid
values across every split after the Section E/F dropna steps, so no additional
imputation is required before training.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

---
## H. Save Datasets

In [ ]:
data_final_path = '../../data/processed'

In [ ]:
# Do not modify this code
# Save training set
try:
  X_train.to_csv(f'{data_final_path}/X_train.csv', index=False)
  y_train.to_csv(f'{data_final_path}/y_train.csv', index=False)

  X_val.to_csv(f'{data_final_path}/X_val.csv', index=False)
  y_val.to_csv(f'{data_final_path}/y_val.csv', index=False)

  X_test.to_csv(f'{data_final_path}/X_test.csv', index=False)
  y_test.to_csv(f'{data_final_path}/y_test.csv', index=False)
except Exception as e:
  print(e)

---
## I. Selection of Performance Metrics

In [ ]:
print('Primary metric: RMSE (Root Mean Squared Error)')
print('Secondary metrics: MAE, R-squared')

In [ ]:
performance_metrics_explanations = """
RMSE is the primary metric since CCI is a continuous 0-100 score with a
directly interpretable unit -- an RMSE of 5 means predictions are typically
off by about 5 comfort points, which is directly meaningful to stakeholders
planning outdoor activities. MAE is reported alongside as a metric less
sensitive to outliers, and R-squared shows how much CCI variance the model
explains relative to a naive baseline.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

## J. Train Machine Learning Model

### J.1 Import Algorithm

In [ ]:
from xgboost import XGBRegressor
print(XGBRegressor)

In [ ]:
algorithm_selection_explanations = """
XGBoost Regressor is used: it handles non-linear feature interactions well
(the CCI formula's component scores are individually clipped/non-linear),
requires no feature scaling, handles a mix of continuous weather and
engineered lag/seasonal features naturally, and is a strong, fast baseline
for tabular regression tasks of this size (~4,000 rows).
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### J.2 Set Hyperparameters

In [ ]:
model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

In [ ]:
hyperparameters_selection_explanations = """
A moderate tree depth (5) with a conservative learning rate (0.05) and
enough estimators (300) balances convergence against overfitting on a
dataset of this size (~4,000 rows). Subsampling rows and columns (0.8 each)
adds regularisation to reduce variance. These are reasonable starting values
to be tuned further against validation RMSE if time permits.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### J.3 Fit Model

In [ ]:
model.fit(X_train, y_train)

### J.4 Model Technical Performance

In [ ]:
val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

val_rmse = mean_squared_error(y_val, val_pred, squared=False)
val_mae = mean_absolute_error(y_val, val_pred)
val_r2 = r2_score(y_val, val_pred)

test_rmse = mean_squared_error(y_test, test_pred, squared=False)
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

print(f'Validation -> RMSE: {val_rmse:.2f}  MAE: {val_mae:.2f}  R2: {val_r2:.3f}')
print(f'Test       -> RMSE: {test_rmse:.2f}  MAE: {test_mae:.2f}  R2: {test_r2:.3f}')

In [ ]:
model_performance_explanations = f"""
On the held-out test set, the model achieves an RMSE of {test_rmse:.2f} CCI
points, an MAE of {test_mae:.2f}, and an R-squared of {test_r2:.3f}. Validation
and test metrics are close (RMSE {val_rmse:.2f} vs {test_rmse:.2f}), suggesting
the model is not badly overfit to the training period. An RMSE in this range
means predictions typically land within a few comfort points of the true
3-day-ahead CCI, which is precise enough to meaningfully distinguish
'Comfortable' from 'Excellent' days for planning purposes, though it may
occasionally blur adjacent categories near a band boundary.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### J.5 Business Impact from Current Model Performance

In [ ]:
print('See business_impacts_explanations below.')

In [ ]:
business_impacts_explanations = f"""
With a test RMSE of {test_rmse:.2f}, the model's 3-day CCI forecasts are precise
enough to support outdoor activity planning decisions with reasonable
confidence, though users near a CCI band boundary (e.g. 79 vs 80, the
Comfortable/Excellent cutoff) should treat the prediction as approximate.
The business impact of error here is primarily inconvenience (a mis-planned
event day) rather than safety risk, so this level of accuracy is likely
acceptable for the intended use case, though further tuning could still
reduce planning friction at the margins.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## H. Project Outcomes

In [ ]:
experiment_outcome = 'Hypothesis Confirmed' if test_r2 > 0.5 else 'Hypothesis Partially Confirmed'
print(experiment_outcome)

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [ ]:
experiment_results_explanations = f"""
The experiment shows that CCI can be forecast 3 days ahead from historical
Sydney weather data with an R-squared of {test_r2:.3f} and RMSE of {test_rmse:.2f}
on held-out test data. Key insight: temperature and cloud cover dominate CCI
variance, matching their formula weights, while engineered lag/rolling/
calendar features materially help capture short-term persistence and
seasonality. Further experiments could explore longer lag windows, additional
weather variables beyond the five in the formula (e.g. UV index, dew point),
or ensembling multiple algorithms to improve robustness near CCI band
boundaries.
"""

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)